In [10]:
from sklearn.neighbors import NearestNeighbors
from random import sample
from numpy.random import uniform
import numpy as np
from math import isnan
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [11]:
df = pd.read_csv('movement.csv')
print('Total rows loaded:', len(df))

cols = ['start_lat', 'start_lon', 'end_lat', 'end_lon']
data = df[cols].dropna().reset_index(drop=True)
print('Rows after dropping missing values:', len(data))

data.head()

Total rows loaded: 469810
Rows after dropping missing values: 469810


,start_lat,start_lon,end_lat,end_lon
0,55.658398,12.514628,55.658348,12.515684
1,55.658348,12.515684,55.659286,12.519309
2,55.659286,12.519309,55.677685,12.522237
3,55.677685,12.522237,55.676945,12.520396
4,55.676945,12.520396,55.655346,12.537441


In [12]:
scaler = StandardScaler()
data_scaled = pd.DataFrame(scaler.fit_transform(data), columns=cols)

# Subsample 50,000 randomyl
data_small = data_scaled.sample(n=50000, random_state=42).reset_index(drop=True)


In [13]:

#Reference : Gaurav BR (2023). *Hopkins Statistic Code*. 
# Kaggle. https://www.kaggle.com/code/gauravbr/hopkins-statistic-code-clustering




def hopkins(X, sample_ratio=0.2):
    d = X.shape[1]
    n = len(X)
    m = int(sample_ratio * n)  
    
    print(f'  Rows: {n:,} | Points analysed m: {m:,}')
    
    nbrs = NearestNeighbors(n_neighbors=1).fit(X.values)
    
    rand_X = sample(range(0, n, 1), m)
    
    W_points = X.iloc[rand_X].values
    U_points = uniform(np.amin(X, axis=0), np.amax(X, axis=0), (m, d))
    
    w_dists, _ = nbrs.kneighbors(W_points, 2, return_distance=True)
    wjd = w_dists[:, 1]
    
    u_dists, _ = nbrs.kneighbors(U_points, 2, return_distance=True)
    ujd = u_dists[:, 1]
    
    H = sum(ujd) / (sum(ujd) + sum(wjd))
    
    if isnan(H):
        H = 0
    
    return H


In [15]:
np.random.seed(42)

H = hopkins(data_small)

print(f'\nHopkins Statistic H = {H:.4f}')



  Rows: 50,000 | Points analysed m: 10,000

Hopkins Statistic H = 0.9975
